# Wake Word Benchmark — run every plugin on the arena's test sets

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/OpenVoiceOS/ngi0-notebooks/blob/dev/benchmarks/ww_benchmark.ipynb) [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenVoiceOS/ngi0-notebooks/blob/dev/benchmarks/ww_benchmark.ipynb)

## Start here

This notebook installs every OpenVoiceOS wake-word plugin and gives each one the same clips from the test sets of the [OVOS plugin arena](https://github.com/OpenVoiceOS/ovos-plugin-arena). It counts how often each plugin wakes on its phrase and how often it wakes by mistake. It needs a CPU, about 3 GB of memory and the internet. With the default settings it runs 43 fighters on 9 test sets, and that takes about one and a half hours on a laptop CPU. Press **Run All** and read "What you got" at the bottom. To only look at the arena's published results, use [`ww_leaderboard.ipynb`](ww_leaderboard.ipynb).

<details>
<summary>Send your results to the arena</summary>

The results stay on your machine unless you set `UPLOAD = True` and give a Hugging Face token with write access to the OpenVoiceOS organisation in `HF_TOKEN`. Ask the arena maintainers first. On Kaggle, add the token as a secret called `HF_TOKEN`; never write it into the notebook.

</details>

Change the settings in the Settings cell, and then run all cells. To change a setting without a change to the notebook, set an environment variable with the same name. You can also write the line `NAME=value` in a file called `.env`, in the folder where the notebook runs.

In [ ]:
#@title Code that reads the settings
# A setting comes from an environment variable of the same name, else from a line NAME=value in the file .env,
# else from the value in the Settings cell. A value from outside is turned into the type of that value.
import json, os

_DOTENV = {}
if os.path.isfile(".env"):
    with open(".env", encoding="utf-8") as _f:
        for _line in _f:
            _name, _sep, _value = _line.strip().partition("=")
            if _sep and not _name.startswith("#"):
                _DOTENV[_name.strip()] = _value.strip().strip("\"'")


def setting(name, default):
    raw = os.environ.get(name, _DOTENV.get(name))
    if raw is None:
        return default
    if isinstance(default, bool):
        if raw.strip().lower() in ("1", "true", "yes", "on"):
            return True
        if raw.strip().lower() in ("0", "false", "no", "off", ""):
            return False
        raise ValueError(f"{name} must be True or False, not {raw!r}")
    if isinstance(default, (int, float)):
        try:
            return type(default)(raw)
        except ValueError:
            raise ValueError(f"{name} must be a number, not {raw!r}") from None
    if isinstance(default, (list, dict)):
        return json.loads(raw)
    return raw


In [ ]:
# Settings: change the values below.
DATASETS = setting("DATASETS", ["picovoice-computer", "picovoice-alexa", "picovoice-jarvis", "community-computer", "community-hey-scout", "community-hey-k9", "synthetic-wakewords-hey_mycroft", "synthetic-wakewords-hey_jarvis", "synthetic-wakewords-alexa"])   # arena test sets to run
PLUGINS = setting("PLUGINS", [])                      # plugins to run, such as ["ovos-ww-plugin-precise-onnx"]; empty: every installed plugin
MAX_PER_CLASS = setting("MAX_PER_CLASS", 100)         # clips with the phrase per test set; clips without it come from two pools of this size
UPLOAD = setting("UPLOAD", False)                     # True: send the results to the arena's Hugging Face datasets
HF_TOKEN = setting("HF_TOKEN", "")                    # a Hugging Face token with write access; only for UPLOAD

# Quick test
SMOKE = setting("SMOKE", False)                       # True: one plugin, one test set, a few clips; nothing is uploaded
N_SAMPLES = setting("N_SAMPLES", 4)                   # quick test only: clips with the phrase
# EPOCHS is not used: this notebook trains nothing.

# ── Advanced: leave these as they are unless you know you need them ──
SAME_PHRASE_ONLY = setting("SAME_PHRASE_ONLY", True)  # True: a detector runs only on the test sets of its own phrase
ARENA_REF = setting("ARENA_REF", "72a05099004d652daf9f01a8fa0d08ebf201e2fc")   # the plugin arena commit whose registry, runner and scoring code are used
WORK_DIR = setting("WORK_DIR", "")                    # the folder for downloads; empty: ./ww_benchmark_work, on Kaggle /tmp/ww_benchmark_work
OUTPUT_DIR = setting("OUTPUT_DIR", "./ww_benchmark_output")   # the result rows, the tables and the plots go here
SHOW_DETAILS = setting("SHOW_DETAILS", False)         # True: also print the tables behind every step

In [ ]:
import os

WORK_DIR = WORK_DIR or ("/tmp/ww_benchmark_work" if os.path.isdir("/kaggle") else "./ww_benchmark_work")
SMOKE_FIGHTERS = ["precise-onnx-hey-mycroft"]
if SMOKE:
    DATASETS, MAX_PER_CLASS, UPLOAD = ["synthetic-wakewords-hey_mycroft"], N_SAMPLES, False
    OUTPUT_DIR = OUTPUT_DIR.rstrip("/") + "_smoke"   # the trial's few rows never mix with a real run's rows

## Step 1 — Install the plugins and get the arena code

Installs every wake-word plugin that is on PyPI, and downloads one fixed commit of the plugin arena: its list of plugins and test sets, the code that gives a plugin its clips, and the code that scores it. If a red box appears, read its last line.

<details>
<summary>Plugins that this notebook cannot install</summary>

- `ovos-ww-plugin-pocketsphinx`: the version on PyPI needs an old `pocketsphinx` that does not build. The fixed version is not on PyPI yet.
- `ovos-ww-plugin-wakewordlab`: not on PyPI.

</details>

In [ ]:
#@title Code for step 1: the step lines, the internet check and the arena checkout
import json, os, re, subprocess, sys, time, urllib.error, urllib.request
from pathlib import Path

def detail(*a, **k):
    # SHOW_DETAILS: a line of the tables behind a step, with the clock time
    if SHOW_DETAILS:
        print(time.strftime("%H:%M:%S"), *a, **k)
if not SHOW_DETAILS:
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # no download bars; the step lines say what happened
_STEPS, _last, _done = {}, [time.time()], [time.time()]
def tick(name):
    now = time.time()
    _STEPS[name], _last[0] = now - _last[0], now
def done(name, message):
    # the step's one line, with the clock time and the seconds the step took
    tick(name)
    print(f"{message} [{time.strftime('%H:%M:%S')}, {_last[0] - _done[0]:.0f} s]")
    _done[0] = _last[0]

def check_internet():
    try:
        urllib.request.urlopen("https://huggingface.co", timeout=15)
    except (urllib.error.URLError, OSError) as exc:
        raise SystemExit("No internet. On Kaggle, open the Settings panel and switch Internet on. "
                         f"({exc})") from None

ARENA_URL = "https://github.com/OpenVoiceOS/ovos-plugin-arena"

def fetch_arena(ref, folder):
    # one commit of the plugin arena: its registry, its runner and its scoring code
    folder = Path(folder)
    def git(*args, check=True):
        return subprocess.run(["git", "-C", str(folder), *args], check=check,
                              capture_output=True, text=True).stdout.strip()
    if not (folder / ".git").is_dir():
        folder.mkdir(parents=True, exist_ok=True)
        git("init", "-q")
        git("remote", "add", "origin", ARENA_URL)
    if git("rev-parse", "-q", "--verify", "HEAD", check=False) != ref:
        git("fetch", "-q", "--depth", "1", "origin", ref)
        git("checkout", "-q", "--force", "FETCH_HEAD")
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
    return git("rev-parse", "HEAD")

def phrase(text):
    # "hey_mycroft", "Hey Mycroft" and "hey-mycroft" are the same phrase
    return re.sub(r"[\s_\-]+", " ", (text or "").lower()).strip()

# The dataset names its wake-word folder; for these sets the folder name is not the phrase.
DATASET_PHRASE = {"sam-wake-word": "sam"}

def dataset_phrase(ds):
    return phrase(DATASET_PHRASE.get(ds.dataset_id, ds.wakeword))

def fighter_phrase(comp):
    return phrase(next(iter(comp.config.get("hotwords", {})), ""))

In [ ]:
check_internet()
%pip install --quiet --pre "ovoscope>=1.6.0a1" "ovos-plugin-manager>=2.0.0" "pydantic>=2" "huggingface_hub>=0.30" "numpy>=1.26" "soundfile>=0.12" "scipy>=1.11" "av>=12" "pyarrow>=15" "pandas>=2" "matplotlib>=3.8" "ovos-ww-plugin-precise-onnx>=0.1.2a1" "ovos-ww-plugin-microwakeword>=0.0.1a4" "ovos-ww-plugin-vosk>=0.1.11a3" "ovos-ww-plugin-wakeforge>=0.18.0a1" "ovos-ww-plugin-efficientword-net>=0.0.1a2" "ovos-ww-plugin-local-wake>=0.0.1a2" "ovos-vad-plugin-silero>=0.1.3a2" "ovos-ww-verifier-plugin-speaker>=0.0.4a3"
# openWakeWord declares tflite-runtime, which has no wheel for new Python versions; its plugin documents this two-step install
%pip install --quiet --pre --no-deps "openwakeword>=0.5.0,<1"
%pip install --quiet --pre "ovos-ww-plugin-openwakeword[onnx]>=0.4.6"
ARENA_DIR = Path(WORK_DIR) / "ovos-plugin-arena"
ARENA_SHA = fetch_arena(ARENA_REF, ARENA_DIR)
from importlib.metadata import entry_points
INSTALLED = sorted({e.name for g in ("opm.wake_word", "mycroft.plugin.wake_word") for e in entry_points(group=g)})
detail("installed wake-word plugins:", INSTALLED)
done("install", f"Step 1 done: {len(INSTALLED)} wake-word plugins installed, plugin arena at commit {ARENA_SHA[:10]}.")

## Step 2 — Choose the plugins for each test set

A **fighter** is one plugin with one set of settings, as the arena lists it. Each test set gets the fighters for its phrase. A fighter that trained on a test set does not run on it.

Three kinds of fighter need no trained model for their phrase:

- `ovos-ww-plugin-wakeforge-zeroshot` gets the phrase as phones, made once with eSpeak NG.
- `ovos-ww-plugin-efficientword-net` uses its own recordings of "alexa" and "computer".
- `ovos-ww-plugin-local-wake` gets four recordings of the phrase from one test set, and it does not run on that set.

In [ ]:
#@title Code for step 2: the fighters for each test set
import gc, logging
logging.getLogger().setLevel(logging.INFO if SHOW_DETAILS else logging.ERROR)

from registry.loaders import list_competitors, list_datasets, results_repo_for
from runner.intent_bench import resolve_revision
from runner.media_bench import plugin_is_installed, run_competitor_lang
from runner.ww_bench import WakeWordBench

REGISTRY = list_competitors("wake_word")
ALL_SETS = {d.dataset_id: d for d in list_datasets("wake_word")}

def fighters_for(ds):
    # ([fighter], [(fighter id, why it does not run)])
    chosen, skipped = [], []
    for comp in REGISTRY:
        if SMOKE and comp.competitor_id not in SMOKE_FIGHTERS:
            continue
        if PLUGINS and comp.plugin not in PLUGINS:
            continue
        if SAME_PHRASE_ONLY and fighter_phrase(comp) != dataset_phrase(ds):
            continue
        if ds.dataset_id in (comp.trained_on or []):
            skipped.append((comp.competitor_id, "trained on this test set"))
        elif not plugin_is_installed("wake_word", comp.plugin):
            skipped.append((comp.competitor_id, f"{comp.plugin} is not installed"))
        else:
            chosen.append(comp)
    return chosen, skipped

In [ ]:
unknown = sorted(set(DATASETS) - set(ALL_SETS))
assert not unknown, f"DATASETS names {unknown}, which the arena does not know. Known: {sorted(ALL_SETS)}"
SETS = [ALL_SETS[d] for d in DATASETS]
PAIRS, SKIPPED = [], []
for ds in SETS:
    chosen, skipped = fighters_for(ds)
    PAIRS += [(ds, comp) for comp in chosen]
    SKIPPED += [(ds.dataset_id, cid, why) for cid, why in skipped]
    detail(ds.dataset_id, [c.competitor_id for c in chosen])
assert PAIRS, "No plugin matches these test sets and PLUGINS."
done("fighters", f"Step 2 done: {len(PAIRS)} fighter runs on {len(SETS)} test sets, {len(SKIPPED)} skipped.")

## Step 3 — Run the plugins

Each clip goes to the plugin in pieces of 80 ms, the way the OpenVoiceOS listener gives it the microphone. The result is one row per clip: did the clip have the phrase, and did the plugin wake. The rows go to `ww_benchmark_output/rows/`. A run that stops can start again: the rows already written are kept, and only the missing clips run.

In [ ]:
#@title Code for step 3: one run per fighter and test set, with the arena's runner
BENCH = WakeWordBench()
ROWS = Path(OUTPUT_DIR) / "rows"
REVISIONS = {}

def rows_path(ds, comp):
    return ROWS / ds.dataset_id / "wake_word" / "predictions" / ds.lang / f"{comp.competitor_id}.jsonl"

def run_pair(ds, comp):
    if ds.dataset_id not in REVISIONS:
        REVISIONS[ds.dataset_id] = resolve_revision(ds.source.hf_id, ds.source.revision)
    path = rows_path(ds, comp)
    path.parent.mkdir(parents=True, exist_ok=True)
    result = run_competitor_lang(BENCH, comp, ds.dataset_id, ds.lang, ds, REVISIONS[ds.dataset_id], path,
                                 ROWS / ds.dataset_id / "wake_word" / "audio",
                                 results_repo_for("wake_word", ds.dataset_id), max_samples=MAX_PER_CLASS)
    gc.collect()
    return result

In [ ]:
RUNS, FAILED = {}, {}
for i, (ds, comp) in enumerate(PAIRS, 1):
    started = time.time()
    try:
        RUNS[(ds.dataset_id, comp.competitor_id)] = run_pair(ds, comp)
    except Exception as exc:   # one broken plugin must not stop the others
        FAILED[(ds.dataset_id, comp.competitor_id)] = f"{type(exc).__name__}: {exc}"
    detail(f"{i}/{len(PAIRS)}", ds.dataset_id, comp.competitor_id,
           RUNS.get((ds.dataset_id, comp.competitor_id)) or FAILED[(ds.dataset_id, comp.competitor_id)],
           f"{time.time() - started:.0f} s")
written = sum(r.written for r in RUNS.values())
errored = sum(r.errored for r in RUNS.values())
done("run", f"Step 3 done: {len(RUNS)} of {len(PAIRS)} fighter runs finished, {written} new rows, "
     f"{errored} clips failed" + (f", {len(FAILED)} runs failed: {sorted(FAILED)}" if FAILED else "") + ".")

## Step 4 — Score the results

The arena's scoring code turns the rows into one board per test set, the same board the arena shows. The **error rate** is the share of clips the plugin got wrong. A false reject is a clip with the phrase that did not wake the plugin. A false accept is a clip without the phrase that woke it. Lower is better. The notebook also counts the mistakes itself from the rows, and the column `own count agrees` says whether its count gives the same error rate.

In [ ]:
#@title Code for step 4: one board per test set, and an own count of the mistakes
import pandas as pd
from arena.metrics import MIN_BOARD_SAMPLES, build_benchmark_board
from arena.predictions import read_jsonl

def own_error_rate(path):
    # wrong clips / all clips, counted from the raw rows without the arena's code
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    wrong = sum((r["label"] == "positive") != (r["prediction"] == "detected") for r in rows)
    return wrong / len(rows) if rows else None

def published_entry(ds, competitor_id):
    path = ARENA_DIR / "frontend-static/public/data" / f"benchmark-wake_word-{ds.dataset_id}-{ds.lang}.json"
    if not path.is_file():
        return {}
    return next((e for e in json.loads(path.read_text())["entries"] if e["competitor_id"] == competitor_id), {})

def score_set(ds):
    mine = {comp.competitor_id: comp for d, comp in PAIRS if d is ds}
    paths = {cid: rows_path(ds, comp) for cid, comp in mine.items()}
    by_fighter = {cid: rows for cid, path in paths.items() if path.is_file() and (rows := read_jsonl(path))}
    if not by_fighter:
        return []
    board = build_benchmark_board("wake_word", ds.dataset_id, ds.lang, by_fighter, "notebook",
                                  dataset_revision=ds.source.revision)
    table = []
    for e in board.entries:
        comp = mine[e.competitor_id]
        m, published = e.metrics, published_entry(ds, e.competitor_id)
        positives = sum(1 for r in by_fighter[e.competitor_id] if r.label == "positive")
        own = own_error_rate(paths[e.competitor_id])
        table.append({
            "test set": ds.dataset_id, "fighter": e.competitor_id, "plugin": comp.plugin, "clips": e.samples, "with phrase": positives,
            "without phrase": e.samples - positives, "error rate": m.get("error_rate"),
            "error low": e.primary_metric_ci_lower, "error high": e.primary_metric_ci_upper,
            "false accepts": m.get("false_accept_rate"), "false rejects": m.get("false_reject_rate"),
            "false wake-ups per hour": m.get("fa_per_hour"), "latency ms": m.get("latency_ms_median"),
            "ranked": not e.unranked, "own count agrees": own is not None and round(own, 4) == m.get("error_rate"),
            "published clips": published.get("samples"),
            "published error rate": (published.get("metrics") or {}).get("error_rate"),
        })
    return table

In [ ]:
RESULTS = pd.DataFrame([row for ds in SETS for row in score_set(ds)])
assert len(RESULTS), "No fighter wrote a result row. Set SHOW_DETAILS = True and run again to see why."
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
RESULTS.to_csv(Path(OUTPUT_DIR) / "ww_benchmark.csv", index=False)
disagree = RESULTS[~RESULTS["own count agrees"]]
if SHOW_DETAILS:
    print(RESULTS.to_string(index=False))
done("score", f"Step 4 done: {len(RESULTS)} results, {int(RESULTS['ranked'].sum())} with {MIN_BOARD_SAMPLES} or more clips; "
     + (f"the own count differs for {list(disagree['fighter'])}." if len(disagree) else "the own count agrees with every one."))

## Step 5 — Plots

The first plot shows each result as a point. A point to the left wakes less by mistake. A point lower down misses the phrase less. A filled point has 30 or more clips. The second plot shows how long each plugin takes to decide on one clip.

In [ ]:
#@title Code for step 5: the plots
import matplotlib
import matplotlib.pyplot as plt

def plot_errors(table, path):
    fig, ax = plt.subplots(figsize=(8, 6))
    colours = dict(zip(sorted(table["plugin"].unique()), matplotlib.colormaps["tab10"].colors))
    for plugin, part in table.groupby("plugin"):
        ax.scatter(part["false accepts"], part["false rejects"], s=60, label=plugin, edgecolors=colours[plugin],
                   facecolors=[colours[plugin] if r else "none" for r in part["ranked"]])
    ax.set_xlabel("false accepts (share of clips without the phrase that woke it)")
    ax.set_ylabel("false rejects (share of clips with the phrase it missed)")
    ax.set_xlim(-0.03, 1.03); ax.set_ylim(-0.03, 1.03)
    ax.legend(fontsize=7); ax.set_title("Wake-word plugins: mistakes per test set")
    fig.tight_layout(); fig.savefig(path, dpi=120)
    return fig

def plot_latency(table, path):
    fig, ax = plt.subplots(figsize=(8, 4))
    per = table.groupby("plugin")["latency ms"].median().sort_values()
    ax.barh(per.index, per.values)
    ax.set_xlabel("median time to decide on one clip (ms)")
    fig.tight_layout(); fig.savefig(path, dpi=120)
    return fig

In [ ]:
plot_errors(RESULTS, Path(OUTPUT_DIR) / "ww_errors.png"); plt.show()
plot_latency(RESULTS, Path(OUTPUT_DIR) / "ww_latency.png"); plt.show()
done("plots", f"Step 5 done: plots saved in {OUTPUT_DIR}.")

## Step 6 — Send the results to the arena (only with `UPLOAD = True`)

This step does nothing unless `UPLOAD` is `True`. Then it sends the rows to the arena's Hugging Face datasets, one dataset per test set. First it adds the rows that are already published and that this run did not make, so no published row is lost. It sends a test set only when every fighter on it has 30 or more clips.

In [ ]:
#@title Code for step 6: merges the published rows into this run's rows, then uploads
from arena.predictions import fetch_hf_predictions
from runner.media_bench import upload_predictions

def merge_published(ds):
    # appends to each local file the published rows of the same fighter whose clip is not in it
    published = fetch_hf_predictions(results_repo_for("wake_word", ds.dataset_id))
    added = 0
    for path in (ROWS / ds.dataset_id / "wake_word" / "predictions" / ds.lang).glob("*.jsonl"):
        remote = Path(published) / ds.lang / path.name if published else None
        if not remote or not remote.is_file():
            continue
        have = {json.loads(line)["sample_id"] for line in path.read_text().splitlines() if line.strip()}
        extra = [line for line in remote.read_text().splitlines()
                 if line.strip() and json.loads(line)["sample_id"] not in have]
        with path.open("a") as fh:
            fh.writelines(line + "\n" for line in extra)
        added += len(extra)
    return added

def ready_to_upload(ds):
    mine = RESULTS[RESULTS["test set"] == ds.dataset_id]
    return len(mine) and bool((mine["clips"] >= MIN_BOARD_SAMPLES).all())

In [ ]:
UPLOADED, NOT_UPLOADED = [], []
if UPLOAD:
    if not HF_TOKEN and os.path.isdir("/kaggle"):
        try:
            from kaggle_secrets import UserSecretsClient
            HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    assert HF_TOKEN, "UPLOAD is True but HF_TOKEN is empty. Give a token with write access to the OpenVoiceOS organisation."
    os.environ["HF_TOKEN"] = HF_TOKEN   # huggingface_hub reads the token from here
    for ds in SETS:
        if not ready_to_upload(ds):
            NOT_UPLOADED.append(ds.dataset_id)
            continue
        added = merge_published(ds)
        repo = results_repo_for("wake_word", ds.dataset_id)
        print(f"Uploading {ds.dataset_id} to {repo} ({added} published rows kept)")
        upload_predictions(BENCH, ROWS / ds.dataset_id, ds.dataset_id, ds)
        UPLOADED.append(repo)
done("upload", "Step 6 done: " + (f"{len(UPLOADED)} test sets uploaded"
     + (f", not uploaded because a fighter has fewer than {MIN_BOARD_SAMPLES} clips: {NOT_UPLOADED}" if NOT_UPLOADED else "")
     if UPLOAD else "nothing uploaded, because UPLOAD is False") + ".")

## What you got

**The table** below lists the best result on each test set. The whole table is in `ww_benchmark_output/ww_benchmark.csv`, and the result rows are in `ww_benchmark_output/rows/`. **The plots** are `ww_errors.png` and `ww_latency.png` in the same folder.

Read the error rate together with the number of clips. A result with fewer than 30 clips is not ranked. The columns `published clips` and `published error rate` show what the arena published for the same fighter, so you can see how this run compares. Synthetic voices are easier than people. The `picovoice` sets are the fairest test.

<details>
<summary>Read more</summary>

- [The plugin arena](https://github.com/OpenVoiceOS/ovos-plugin-arena), its [methodology](https://github.com/OpenVoiceOS/ovos-plugin-arena/blob/dev/docs/methodology.md), and [how to add a fighter](https://github.com/OpenVoiceOS/ovos-plugin-arena/blob/dev/docs/add-a-fighter.md)
- [Wake word plugins](https://openvoiceos.github.io/ovos-technical-manual/312-wake_word_plugins/) in the OVOS manual
- [`docs/benchmarks.md`](../docs/benchmarks.md), the guide to these notebooks

</details>

In [ ]:
best = RESULTS.sort_values(["ranked", "error rate"], ascending=[False, True]).groupby("test set").head(1)
columns = ["test set", "fighter", "clips", "error rate", "false accepts", "false rejects"]
print(f"""Fighter runs          : {len(RUNS)} of {len(PAIRS)} finished{', failed: ' + ', '.join(f'{d}/{c}' for d, c in FAILED) if FAILED else ''}
Skipped               : {len(SKIPPED)}{' (' + '; '.join(f'{c} on {d}: {w}' for d, c, w in SKIPPED[:8]) + (' ...' if len(SKIPPED) > 8 else '') + ')' if SKIPPED else ''}
Results               : {len(RESULTS)}, {int(RESULTS['ranked'].sum())} with enough clips to rank
Own count agrees      : {int(RESULTS['own count agrees'].sum())} of {len(RESULTS)}
Uploaded              : {', '.join(UPLOADED) or 'nothing'}
Arena commit          : {ARENA_SHA}
Minutes per step      : {', '.join(f'{k} {v / 60:.1f}' for k, v in _STEPS.items())}

Best result per test set:
{best[columns].to_string(index=False)}""")
if SMOKE:   # the trial must run a real plugin on clips of both kinds and agree with the arena's score
    row = RESULTS.iloc[0]
    assert not FAILED and len(RESULTS) == 1, f"the trial run failed: {FAILED}"
    assert row["with phrase"] > 0 and row["without phrase"] > 0, "the trial needs clips with and without the phrase"
    assert row["own count agrees"], "the own count of mistakes differs from the arena's error rate"
    assert row["false rejects"] < 1, "the plugin did not wake on any clip with its phrase"

[![NGI0 Commons Fund](../ngi.png)](https://nlnet.nl/project/OpenVoiceOS)

This project was funded through the [NGI0 Commons Fund](https://nlnet.nl/commonsfund), a fund established by [NLnet](https://nlnet.nl) with financial support from the European Commission's [Next Generation Internet](https://ngi.eu) programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No [101135429](https://cordis.europa.eu/project/id/101135429).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

Licensed under Apache-2.0.